## Setup del Notebook

In [20]:
from pathlib import Path
from time import gmtime, strftime
import boto3
import pandas as pd
import sagemaker
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput
from sagemaker.workflow.steps import ProcessingStep

In [21]:
# Definimos la raíz del repositorio y configuramos la sesión de SageMaker
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Configuramos la sesión de SageMaker
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = sagemaker_session.default_bucket()
region = boto3.session.Session().region_name
timestamp = strftime("%Y-%m-%d-%H-%M-%S", gmtime())

# Definimos el prefijo de S3 y las rutas de entrada y salida
prefix = f"sagemaker/tarea06-processing-byoc/{timestamp}"
input_prefix = f"{prefix}/input/raw"
output_prefix = f"{prefix}/output/preprocessed"

raw_s3_uri = f"s3://{bucket}/{input_prefix}/"
processed_s3_uri = f"s3://{bucket}/{output_prefix}/"

print("REPO_ROOT:", REPO_ROOT)
print("raw_s3_uri:", raw_s3_uri)
print("processed_s3_uri:", processed_s3_uri)

REPO_ROOT: /home/sagemaker-user/Tarea3_ProductoDeDatos
raw_s3_uri: s3://sagemaker-us-east-1-110276528929/sagemaker/tarea06-processing-byoc/2026-03-22-11-36-14/input/raw/
processed_s3_uri: s3://sagemaker-us-east-1-110276528929/sagemaker/tarea06-processing-byoc/2026-03-22-11-36-14/output/preprocessed/


In [22]:
# Carga del dataset a S3
s3 = boto3.client("s3")
for filename in ["sales_train.csv", "test.csv"]:
    local_path = REPO_ROOT / "data" / "raw" / filename
    s3.upload_file(str(local_path), bucket, f"{input_prefix}/{filename}")

print("Raw files uploaded.")

Raw files uploaded.


## ProcessingStep (preprocessing) con imagen BYOC de ECR

In [23]:
# Preparación de la imagen (ya debes tener la imagen construida y subida a ECR)
account_id = boto3.client("sts").get_caller_identity()["Account"]
ecr_repository = "tarea06-processing-byoc"
image_uri = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{ecr_repository}:latest"

# Configuración del contenedor de procesamiento con ScriptProcessor
script_processor = ScriptProcessor(
    image_uri=image_uri, # URI de tu imagen de ECR
    command=["python3"],
    instance_type="ml.m5.xlarge", # Tipo de instancia para el procesamiento
    instance_count=1, # Número de instancias
    role=role, # El rol de IAM con los permisos adecuados
    sagemaker_session=sagemaker_session,
)

# 5. Ejecutamos el preprocesamiento con el ScriptProcessor
step_process_args = script_processor.run(
    inputs=[ProcessingInput(
        source=raw_s3_uri, # Los datos de entrada desde S3
        destination="/opt/ml/processing/input/raw"
    )],
    outputs=[
        ProcessingOutput(output_name="train", source="/opt/ml/processing/train"),
        ProcessingOutput(output_name="validation", source="/opt/ml/processing/validation"),
        ProcessingOutput(output_name="test", source="/opt/ml/processing/test"),
    ],
    code="processing/preprocess.py", # Ruta al script de procesamiento en tu repositorio
)

# 6. Definimos el ProcessingStep para el pipeline
step_process = ProcessingStep(
    name="PreprocessingStep", # Nombre del step en el pipeline
    step_args=step_process_args
)

# Para la verificación, podemos imprimir el nombre del job y verificar el estado
job_name = script_processor.latest_job.job_name
print("Job Name:", job_name)

# Verificamos el estado y los archivos de salida
desc = sagemaker_session.sagemaker_client.describe_processing_job(
    ProcessingJobName=job_name
)

print("Status:", desc["ProcessingJobStatus"])
print("Failure Reason:", desc.get("FailureReason"))

# Mostramos los archivos de salida generados por el step de procesamiento
for o in desc["ProcessingOutputConfig"]["Outputs"]:
    print(o["OutputName"], "->", o["S3Output"]["S3Uri"])

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:17                                                                                   │
│                                                                                                  │
│   14 )                                                                                           │
│   15                                                                                             │
│   16 # 5. Ejecutamos el preprocesamiento con el ScriptProcessor                                  │
│ ❱ 17 step_process_args = script_processor.run(                                                   │
│   18 │   inputs=[ProcessingInput(                                                                │
│   19 │   │   source=raw_s3_uri, # Los datos de entrada desde S3                                  │
│   20 │   │   destination="/opt/ml/processing/input/raw"                                          │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:346 in wrapper    │
│                                                                                                  │
│   343 │   │   │                                                                                  │
│   344 │   │   │   return _StepArguments(retrieve_caller_name(self_instance), run_func, *args,    │
│   345 │   │                                                                                      │
│ ❱ 346 │   │   return run_func(*args, **kwargs)                                                   │
│   347 │                                                                                          │
│   348 │   return wrapper                                                                         │
│   349                                                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/processing.py:670 in run                       │
│                                                                                                  │
│    667 │   │   │   None or pipeline step arguments in case the Processor instance is built with  │
│    668 │   │   │   :class:`~sagemaker.workflow.pipeline_context.PipelineSession`                 │
│    669 │   │   """                                                                               │
│ ❱  670 │   │   normalized_inputs, normalized_outputs = self._normalize_args(                     │
│    671 │   │   │   job_name=job_name,                                                            │
│    672 │   │   │   arguments=arguments,                                                          │
│    673 │   │   │   inputs=inputs,                                                                │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/processing.py:319 in _normalize_args           │
│                                                                                                  │
│    316 │   │                                                                                     │
│    317 │   │   self._current_job_name = self._generate_current_job_name(job_name=job_name)       │
│    318 │   │                                                                                     │
│ ❱  319 │   │   inputs_with_code = self._include_code_in_inputs(inputs, code, kms_key)            │
│    320 │   │   normalized_inputs = self._normalize_inputs(inputs_with_code, kms_key)             │
│    321 │   │   normalized_outputs = self._normalize_outputs(outputs)                             │
│    322 │   │   self.arguments = arguments                  